In [1]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\civil\bimeh.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\civil\bimeh_articles.tsv"

LAW_CODE = "bimeh_ejbari"
LAW_NAME = "قانون بیمه اجباری خسارات واردشده به شخص ثالث در اثر حوادث ناشی از وسایل نقلیه"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans_table = str.maketrans(persian_digits, english_digits)
    return text.translate(trans_table)


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن و نرمال‌سازی اولیه
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-").replace("ـ", "-")

# 2) شکستن خطوط: قبل از «بخش» و «ماده» در وسط خط
# الگوی بخش: "بخش نخست-" یا "بخش سوم-"
raw = re.sub(r"(?<!^)(بخش\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)

# الگوی ماده با انعطاف بیشتر: "ماده1–" یا "ماده ۲۱ –"
# توجه: نیم‌فاصله‌ها و انواع dash را هم پوشش می‌دهیم
raw = re.sub(r"(?<!^)(ماده\s*‌?\s*\d+\s*[–\-ـ]\s*)", r"\n\1", raw, flags=re.MULTILINE)

lines = raw.splitlines()

# 3) الگوها: بخش و ماده
# مثال بخش: "بخش نخست- کلیات"
section_pattern = re.compile(r"^\s*بخش\s+(.+)$")

# مثال ماده: "ماده۱–" یا "ماده ۲۱ –"
article_pattern = re.compile(r"^\s*ماده\s*‌?\s*(\d+)\s*[–\-ـ]\s*(.*)$")

current_section = ""
articles = []

current_article_num = None
current_article_text = ""

for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # 1) تشخیص بخش
    m_sec = section_pattern.match(stripped)
    if m_sec:
        # نهایی کردن ماده باز
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "section": current_section,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_section = normalize_persian(stripped)
        continue

    # 2) تشخیص شروع ماده
    m_art = article_pattern.match(stripped)
    if m_art:
        # نهایی کردن ماده قبلی
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "section": current_section,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )

        num_str = m_art.group(1)
        rest_text = m_art.group(2).strip()

        try:
            current_article_num = int(num_str)
        except ValueError:
            current_article_num = None

        if rest_text:
            current_article_text = f"ماده {num_str}- {rest_text}"
        else:
            current_article_text = f"ماده {num_str}-"
        continue

    # 3) ادامهٔ متن مادهٔ جاری
    if current_article_num is not None:
        current_article_text += " " + stripped

# 4) بستن آخرین ماده
if current_article_num is not None and current_article_text:
    one_line = normalize_persian(current_article_text)
    articles.append(
        {
            "law_code": LAW_CODE,
            "law_name": LAW_NAME,
            "section": current_section,
            "article_number": current_article_num,
            "text": one_line,
        }
    )

# 5) نوشتن خروجی TSV
fieldnames = ["law_code", "law_name", "section", "article_number", "text"]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش قانون بیمه تمام شد.")
print(f"✓ تعداد مواد استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش قانون بیمه تمام شد.
✓ تعداد مواد استخراج شده: 66
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\civil\bimeh_articles.tsv
